# Actividad 1 — Auditoría y preparación de un corpus

Valor: 8% de la materia.

Una organización ficticia recibe solicitudes de soporte en español. Antes de utilizar estos mensajes para desarrollar un clasificador, necesita un corpus confiable y documentado. Actúas como analista de datos: debes recibir el corpus, auditarlo, decidir qué conservar o modificar, preparar una versión reproducible y separar los datos para un uso posterior.

El producto de esta actividad es un corpus preparado y documentado. No se entrena ningún modelo ni se calculan predicciones o métricas de clasificación.

## Objetivo
Aplicar una auditoría a textos imperfectos, diseñar una preparación adecuada para español y justificar, con ejemplos verificables, qué información conservas, modificas o excluyes. Cada transformación importante debe conectar problema, decisión, implementación, justificación y comprobación.

## Lo que ya debes poder utilizar
De Clase 1: entrada y salida, corpus, etiquetas provisionales, contexto, ambigüedad y negación. De Clase 2: DataFrame, faltantes, duplicados, Unicode/NFC, UTF-8, normalización, regex básica, tokenización, privacidad, familias, particiones y reconstrucción desde un kernel nuevo.

Tecnologías: Python, pandas y spaCy; re y unicodedata forman parte de Python. Jupyter local o Google Colab, con CPU. Para tokenizar usamos spacy.blank("es"), ya explicado en la práctica de Clase 2: no descarga un modelo entrenado y no obtiene lemas. No se exige usar también NLTK ni repetir el trabajo con otra biblioteca.

Puedes consultar las presentaciones y notebooks de Clases 1 y 2 durante toda la actividad. Los pasos y operaciones de apoyo están indicados en cada sección; las decisiones sobre este corpus son tuyas.

## Datos y diccionario
Usa exclusivamente data/corpus_auditoria_v2.csv. Es un CSV en UTF-8 con textos sintéticos autorizados para docencia. Consulta también data/README.md.

| Campo | Significado y tratamiento |
| --- | --- |
| id | Identificador estable del registro; no renumerarlo. |
| text | Mensaje original; conservarlo como text_original. |
| label | Categoría provisional: acceso (cuenta, contraseña, autenticación) o pagos (cargos, facturas, recibos, reembolsos). Puede requerir revisión; no inventar etiquetas. |
| family | Procedencia común de variantes relacionadas; mantenerlas juntas en la partición. No es una etiqueta ni una pista para completarla. |

Convención del archivo: `__NULO__` significa dato no recibido; un campo vacío es una cadena vacía. Los espacios también deben inspeccionarse. La carga proporcionada usa `keep_default_na=False` y `na_values=["__NULO__"]` para distinguirlos. Reconocer un faltante no decide automáticamente qué hacer con su fila.

El corpus contiene casos para tomar decisiones; no se especifican sus ubicaciones. No visites los enlaces ni envíes mensajes a los correos: son ficticios. En este ejercicio académico puedes conservar el original en tu entrega, porque todos los datos son sintéticos. Para datos reales, preservar original exige acceso controlado; reemplazar correos no garantiza anonimización total.

## Preparación: elige un entorno
Colab (preferido): abre el notebook Alumno en Google Colab. Sube corpus_auditoria_v2.csv al panel Archivos. Ejecuta la preparación; instala solo las bibliotecas que falten. Al terminar descarga el notebook ejecutado y los archivos de entrega/ antes de cerrar la sesión.

In [9]:
from pathlib import Path
import sys, subprocess, importlib.util, platform, json, re, unicodedata, hashlib
from importlib.metadata import version

# Instala solamente las dos bibliotecas de la actividad si faltan.
faltan = []
if importlib.util.find_spec("pandas") is None:
    faltan.append("pandas==2.2.3")
if importlib.util.find_spec("spacy") is None:
    # Versiones de apoyo compatibles; no son nuevas herramientas de la actividad.
    faltan.extend(["spacy==3.8.7", "typer==0.16.0", "typer-slim==0.16.0", "click==8.1.8"])
elif importlib.util.find_spec("click") is None:
    faltan.extend(["typer==0.16.0", "typer-slim==0.16.0", "click==8.1.8"])
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", *faltan], check=True)
import pandas as pd
import spacy
print({"Python": platform.python_version(), "pandas": pd.__version__,
       "spaCy": spacy.__version__})


{'Python': '3.12.10', 'pandas': '3.0.5', 'spaCy': '3.8.16'}


In [10]:
# Local: encuentra data/ junto al notebook o en una carpeta superior.
# Colab: acepta también el CSV subido al panel Archivos, en el directorio actual.
nombre = "corpus_auditoria_v2.csv"
candidatos = [Path.cwd() / nombre]
candidatos += [base / "data" / nombre for base in [Path.cwd(), *Path.cwd().parents]]
ARCHIVO = next((p for p in candidatos if p.is_file()), None)
if ARCHIVO is None:
    raise FileNotFoundError("Sube corpus_auditoria_v2.csv al panel Archivos o conserva data/ en el paquete extraído.")
raw = pd.read_csv(ARCHIVO, encoding="utf-8", keep_default_na=False,
                  na_values=["__NULO__"], dtype="string")
assert set(raw.columns) == {"id", "text", "label", "family"}
HUELLA_ORIGINAL = hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()
OUT = Path.cwd() / "entrega"
OUT.mkdir(exist_ok=True)
completados = set()
print("Carga lista. Ejecuta tu auditoría antes de transformar.")


Carga lista. Ejecuta tu auditoría antes de transformar.


## 1. Auditar antes de transformar

Define en 2–3 frases la entrada, la salida que entregarás y dos condiciones para considerar confiable el resultado. Carga el archivo sin modificarlo y documenta el diccionario de las cuatro columnas.

Calcula filas, columnas, IDs repetidos, distribución de etiquetas incluyendo ausencias, número de familias y faltantes por columna. Distingue nulo, cadena vacía y solo espacios. Cuenta repeticiones exactas de text indicando si cuentas filas repetidas después de la primera o todas las filas involucradas.

Lee ejemplos: distingue duplicado exacto, variante relacionada y etiqueta discutible. Muestra al menos un caso que requiera revisión de etiqueta o de información insuficiente y una acción justificada. No infieras etiquetas a partir de family. Usa head, isna, duplicated, value_counts y groupby cuando respondan una pregunta concreta; no basta imprimir veinte comandos.

Antecedentes: Clase 1, práctica 1–2; Clase 2, pasos 1–3.

In [11]:
# 1. Importar y analizar tabla
print(" --- Importar y analizar información ---")
df = raw

print("Filas y columnas:", df.shape)
print("IDs repetidos:", int(df["id"].duplicated().sum()))
print("\nEtiquetas:")
print(df["label"].value_counts(dropna=False))
print("\nFamilias únicas:", df["family"].nunique())
print(df["family"].value_counts().describe())
print("\nFilas involucradas en duplicados:", int(df["text"].duplicated(keep=False).sum()))
print("Filas repetidas después de la primera:", int(df["text"].duplicated(keep="first").sum()))
print("\nNulos (__NULO__):\n", df.isna().sum())
print("\nVacíos (''):\n", (df == "").sum())
print("\nSolo espacios:\n", df.apply(lambda c: c.astype(str).str.fullmatch(r"\s+").sum() if c.dtype=="object" else 0))
print("\nConflictos por texto:", int((df.groupby("text")["label"].nunique() > 1).sum()))
print("Conflictos por familia:\n", df.groupby("family")["label"].nunique().sort_values(ascending=False).head())

print("\nDuplicado exacto:")
mask_dup = df["text"].duplicated(keep=False)
print(df[mask_dup][["id","text","label","family"]].head(10))

print("\nVariantes por familia:")
print(df.groupby("family").filter(lambda g: len(g) > 1)[["id","text","label","family"]].head(10))

print("\nEjemplos textos cortos / ambiguos:")
print(df[df["text"].astype(str).str.len() < 15][["id","text","label"]].head(10))

print("\nCasos con misma clave pero etiqueta distinta (dudosos):")
dudosos = df[df.duplicated(subset=["text"], keep=False)].sort_values("text")
print(dudosos[["id", "text", "label", "family"]])

completados.add(1)

 --- Importar y analizar información ---
Filas y columnas: (97, 4)
IDs repetidos: 0

Etiquetas:
label
acceso    49
pagos     47
           1
Name: count, dtype: Int64

Familias únicas: 32
count        32.0
mean      3.03125
std      0.176777
min           3.0
25%           3.0
50%           3.0
75%           3.0
max           4.0
Name: count, dtype: Float64

Filas involucradas en duplicados: 4
Filas repetidas después de la primera: 2

Nulos (__NULO__):
 id        0
text      1
label     0
family    0
dtype: int64

Vacíos (''):
 id        0
text      1
label     1
family    0
dtype: Int64

Solo espacios:
 id        0
text      0
label     0
family    0
dtype: int64

Conflictos por texto: 1
Conflictos por familia:
 family
F030    2
F007    2
F001    1
F018    1
F031    1
Name: label, dtype: int64

Duplicado exacto:
         id                                               text   label family
4   AUD-005  Buen día, el código de verificación no llega a...  acceso   F002
72  AUD-073  Buen d

### Tu análisis

**Entrada:** Un archivo CSV (`corpus_auditoria_v2.csv`) con columnas de `id, text, label, family` que contiene mensajes de soporte.

**Salida:** Un corpus preparado para utilizarse en el ciclo de NLP, con trazabilidad completa.

**Condiciones de confiabilidad:**
1. Cada ID original queda registrado con estado (conservar / cuarentena / duplicado),
   motivo y, en caso de que aplique, el ID equivalente conservado.
2. Los textos originales se conservan intactos en `text_original` y todo el flujo es
   reproducible.


**Diccionario de columnas**
| Campo | Significado | Tratamiento |
|---|---|---|
| `id` | Identificador estable del registro | No renumerar; se conserva tal cual |
| `text` | Mensaje original de soporte | Conservar como `text_original`; se crea `text_processed` aparte |
| `label` | Categoría provisional: `acceso` (cuenta, contraseña, autenticación) o `pagos` (cargos, facturas, recibos, reembolsos) | Puede requerir revisión; no se inventan etiquetas ni se infieren desde `family` |
| `family` | Procedencia común de variantes relacionadas | Mantener la familia completa en una sola partición; no es etiqueta ni pista para completarla |

**Convención de ausencias:** 
- `__NULO__` significa dato no recibido. 
- Un campo vacío (`""`)
es una cadena vacía. 
- Un campo con solo espacios (`"   "`) es un caso distinto.

**Caso dudoso AUD-089 vs AUD-090:** 
Mismo texto, etiquetas distintas (pagos vs acceso). No hay evidencia para decidir cuál es correcta. 

Decisión: ambos van a cuarentena por "etiqueta contradictoria en texto idéntico".



## 2. Demostrar Unicode y decidir una política

Encuentra en el corpus al menos un caso donde NFC cambie la representación interna. Compara dos representaciones pertinentes con repr, len o puntos de código antes y después de unicodedata.normalize("NFC", texto). Explica por qué pueden verse iguales y no comparar igual, qué cambia y qué no cambia. No basta usar solo el ejemplo «á» del profesor.

Escribe una tabla de decisiones que cubra: NFC y espacios; mayúsculas; acentos y ñ; puntuación; números y moneda; emojis; URLs y correos; stopwords y negación. Puedes agrupar operaciones relacionadas. En cada fila indica aplicar o conservar, motivo y una comprobación; una operación que decides no aplicar también es una decisión.

Define el orden de operaciones. No presupongas que mayor cantidad de limpieza es mejor. Compara duplicados con una clave explícita tras revisar NFC y espacios, sin confundir igualdad de texto con igualdad de casos.

Antecedentes: Clase 2, pasos 4–7.

In [12]:
# 1. Buscar casos NFC en el corpus
def inspeccionar(texto):
    if not isinstance(texto, str):
        return False
    return texto != unicodedata.normalize("NFC", texto)

casos_nfc = raw[raw["text"].map(inspeccionar)]
print("Textos del corpus que cambian con NFC:", len(casos_nfc))
if len(casos_nfc) > 0:
    print(casos_nfc[["id", "text"]].head().to_string(index=False))
else:
    print("No hay casos naturales en el corpus.")


# 2. Demostrar NFC con un caso del corpus
caso = casos_nfc.iloc[0]["text"] if len(casos_nfc) > 0 else "contrasen\u0303a"

print("\n- Antes de NFC")
print("repr:", repr(caso))
print("len:", len(caso))
print("puntos de código:", [hex(ord(c)) for c in caso])

nfc = unicodedata.normalize("NFC", caso)
print("\n- Después de NFC")
print("repr:", repr(nfc))
print("len:", len(nfc))
print("puntos de código:", [hex(ord(c)) for c in nfc])
print("¿cambia?", caso != nfc)


# 3. Función de normalización conservadora
def normalizar(texto):
    if not isinstance(texto, str):
        raise TypeError("Se esperaba texto; revisar faltantes antes de normalizar.")
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", texto)).strip()


# 4. Duplicados con clave explícita tras NFC y espacios
def clave(texto):
    if not isinstance(texto, str):
        return None
    return normalizar(texto)

raw["clave"] = raw["text"].astype("object").map(clave)

duplicados_totales = int(raw["clave"].duplicated(keep=False).sum())
duplicados_extra   = int(raw["clave"].duplicated(keep="first").sum())

print("\nFilas involucradas en duplicados por clave:", duplicados_totales)
print("Filas repetidas después de la primera:", duplicados_extra)

# 5. Pruebas de español y negación
print("\n- Pruebas")
casos = {
    "negacion":      "No quiero cancelar mi cuenta.",
    "si_acento":     "sí",
    "si_sin_acento": "si",
    "anio":          "Este año renové mi cuenta.",
    "propio":        "No, quiero cancelar mi cuenta.",  # caso propio
}

for nombre, t in casos.items():
    salida = normalizar(t)
    print(f"{nombre:15} | {t!r} PASA A {salida!r}")
    print(f"{'':15} | ¿cambia? {t != salida}")

# Comprobaciones explícitas
assert normalizar("No quiero cancelar mi cuenta.") == "No quiero cancelar mi cuenta."
assert normalizar("sí") == "sí"
assert normalizar("si") == "si"
assert normalizar("sí") != normalizar("si")
assert normalizar("Este año renové mi cuenta.") == "Este año renové mi cuenta."
assert "No" in normalizar("No, quiero cancelar mi cuenta.")
print("\nTodas las comprobaciones pasaron.")

completados.add(2)

Textos del corpus que cambian con NFC: 1
     id                                                 text
AUD-081 Necesito información para recuperar mi contraseña.

- Antes de NFC
repr: 'Necesito información para recuperar mi contraseña.'
len: 52
puntos de código: ['0x4e', '0x65', '0x63', '0x65', '0x73', '0x69', '0x74', '0x6f', '0x20', '0x69', '0x6e', '0x66', '0x6f', '0x72', '0x6d', '0x61', '0x63', '0x69', '0x6f', '0x301', '0x6e', '0x20', '0x70', '0x61', '0x72', '0x61', '0x20', '0x72', '0x65', '0x63', '0x75', '0x70', '0x65', '0x72', '0x61', '0x72', '0x20', '0x6d', '0x69', '0x20', '0x63', '0x6f', '0x6e', '0x74', '0x72', '0x61', '0x73', '0x65', '0x6e', '0x303', '0x61', '0x2e']

- Después de NFC
repr: 'Necesito información para recuperar mi contraseña.'
len: 50
puntos de código: ['0x4e', '0x65', '0x63', '0x65', '0x73', '0x69', '0x74', '0x6f', '0x20', '0x69', '0x6e', '0x66', '0x6f', '0x72', '0x6d', '0x61', '0x63', '0x69', '0xf3', '0x6e', '0x20', '0x70', '0x61', '0x72', '0x61', '0x20', '0x

### Tu análisis

**Caso NFC del corpus**

El corpus contiene un caso natural donde NFC cambia la representación interna: *AUD-081*. La `ñ` viene descompuesta, pero tras la normalización coincide con la forma compuesta. Ambos casos se ven iguales en pantalla, pero no comparan igual sin NFC dado que la comoputadora. Sin normalizar, dos textos idénticos para un humano no coinciden en `==` ni en `duplicated()`, y los duplicados reales pasan desapercibidos debido a que, internamente, la computadora los reconoce como elementos diferentes debido a su identificador de Unicode.

**Tabla de decisiones**

| Operación | ¿Aplicar o conservar? | Motivo | Comprobación |
|---|---|---|---|
| NFC | Aplicar | Unifica representaciones para que `==` y `duplicated()` sean fiables | AUD-081: `len` 11 → 10 |
| Espacios | Aplicar colapso y `strip()` | Evita duplicados falsos por espacios | `"No   quiero   cancelar."` → `"No quiero cancelar."` |
| Mayúsculas | Conservar | Pueden marcar énfasis o nombres propios | `"NO renovar"` sobrevive |
| Acentos y ñ | Conservar | Cambian significado: `año` ≠ `ano`, `sí` ≠ `si` | `"sí"` ≠ `"si"` tras normalizar |
| Puntuación | Conservar | Afecta negación y sentido | `"No quiero cancelar."` vs `"No, quiero cancelar."` |
| Números y moneda | Conservar | Señal para `pagos` | `"$1,250.50"` intacto |
| Emojis | Conservar | Pueden aportar tono | Sin cambios |
| URLs y correos | Reemplazar por `[URL]` / `[CORREO]` | Privacidad | Regex aplicada en 3 ejemplos |
| Stopwords | Conservar | Borrar `no` cambia la tarea | `"No quiero cancelar."` conserva `No` |
| Negación | Conservar siempre | Invierte el sentido | Prueba con `"No quiero cancelar mi cuenta."` |

**Orden de operaciones**

1. NFC.
2. Colapso de espacios y `strip()`.
3. Reemplazo de URLs y correos.

No se aplicó `lower()`, ni se eliminaron acentos o stopwords. Ya que para este caso pueden resultar útiles para entender el contexto o intención general del mensaje. 

**Límites del regex**

El patrón de correo no reconoce todos los correos válidos ni anonimiza nombres, teléfonos o contexto. Reemplazar un correo no garantiza anonimización total; por eso se conserva `text_original`.

**Pruebas de español**

| Assert	| Qué comprueba|
| ---|---|
|normalizar("No quiero...") == "No quiero..."	|La negación "No" sobrevive.|
|normalizar("sí") == "sí"|	El acento sobrevive.|
|normalizar("si") == "si"|La forma sin acento no se altera.|
|normalizar("sí") != normalizar("si")	|El acento distingue significado.|
|normalizar("Este año...") == "Este año..."	|La ñ y el acento sobreviven.|
|"No" in normalizar("No, quiero...")	|La negación sobrevive incluso con coma.|

**Resultado**

Encontré un caso NFC en el corpus (AUD-081) y observé que dos representaciones visualmente idénticas no comparan igual sin normalizar. La política aplica NFC y colapso de espacios, y conserva acentos, ñ, mayúsculas, puntuación, números, emojis y negación. Las pruebas confirman que `"No"`, `"sí"` y `"año"` se conservan por su valor de significado e intención. Tras la clave explícita aparecen 8 filas involucradas en duplicados (4 después de la primera), lo que confirma que revisar NFC antes de declarar duplicados es necesario.

## 3. Preparar sin perder trazabilidad

Implementa tu política como función y aplícala a una copia. Conserva text_original sin alteraciones y crea text_processed. No edites el CSV a mano. Decide el tratamiento de ausencias, duplicados, etiquetas dudosas y textos insuficientes. Puedes poner registros en cuarentena (apartados para revisión, sin usarlos todavía) en vez de resolver una duda sin evidencia.

Lleva un registro con una fila por ID original: estado (conservar, cuarentena o duplicado), motivo y, cuando corresponda, ID del registro conservado equivalente. No elimines filas sin dejar rastro. No cambies una etiqueta por adivinación; una corrección necesita evidencia y política explícitas.

Identifica información potencialmente sensible y aplica o justifica una política para ella. Basta una regex sencilla de correos, URLs o espacios; puedes adaptar la de Clase 2. Explica sus límites. No tienes que desarrollar un detector universal.

Muestra una tabla antes/después de al menos seis filas pertinentes del corpus, incluyendo cambios y decisiones de conservar. Comprueba que sobreviven acentos, ñ y negación cuando afectan el sentido. Prueba también estos contrastes: «No quiero cancelar mi cuenta.»; «sí» frente a «si»; «Este año renové mi cuenta.». Añade un caso propio que pueda romper tu regla. No cambies la fuente para superar una comprobación.

Antecedentes: Clase 2, pasos 5–7 y 12.

In [13]:
# 1. Función de preparación
def preparar(texto):
    if not isinstance(texto, str):
        return None
    texto = unicodedata.normalize("NFC", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

# 2. Copia con original y procesado
corpus = raw.copy()
corpus["text_original"]  = corpus["text"]
corpus["text_processed"] = corpus["text_original"].map(preparar)

# 3. Política de información sensible
patron_correo = r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}"
patron_url    = r"https?://\S+|www\.\S+"

def enmascarar(texto):
    if not isinstance(texto, str):
        return texto
    texto = re.sub(patron_correo, "[CORREO]", texto)
    texto = re.sub(patron_url, "[URL]", texto)
    return texto

corpus["text_processed"] = corpus["text_processed"].map(enmascarar)

# 4. Estado por ID
vistos = {}
estado, motivo, equivalente = [], [], []

for _, row in corpus.iterrows():
    t = row["text_original"]
    if not isinstance(t, str) or t.strip() == "":
        estado.append("cuarentena"); motivo.append("texto ausente"); equivalente.append(None)
    elif len(t.strip()) < 10:
        estado.append("cuarentena"); motivo.append("texto insuficiente"); equivalente.append(None)
    elif not isinstance(row["label"], str) or row["label"].strip() == "":
        estado.append("cuarentena"); motivo.append("etiqueta ausente"); equivalente.append(None)
    elif row["text_processed"] in vistos:
        estado.append("duplicado"); motivo.append("clave ya vista"); equivalente.append(vistos[row["text_processed"]])
    else:
        vistos[row["text_processed"]] = row["id"]
        estado.append("conservar"); motivo.append("ok"); equivalente.append(None)

corpus["estado"]         = estado
corpus["motivo"]         = motivo
corpus["id_equivalente"] = equivalente

# 5. Registro de decisiones
registro = corpus[["id", "estado", "motivo", "id_equivalente"]].rename(columns={"id": "id_original"})

# 6. Tabla antes/después
muestra = corpus[
    (corpus["text_original"] != corpus["text_processed"]) |
    (corpus["estado"] != "conservar")
].head(6)
print("Tabla antes | después: ")
print(muestra[["id", "text_original", "text_processed", "estado", "motivo"]].to_string(index=False))

# 7. Pruebas
print("\nPruebas: ")
casos = {
    "negacion":      "No quiero cancelar mi cuenta.",
    "si_acento":     "sí",
    "si_sin_acento": "si",
    "anio":          "Este año renové mi cuenta.",
    "propio":        "No, quiero cancelar mi cuenta.",
}
for nombre, t in casos.items():
    print(f"{nombre:15} | {t!r} PASA A {preparar(t)!r}")

assert preparar("No quiero cancelar mi cuenta.") == "No quiero cancelar mi cuenta."
assert preparar("sí") != preparar("si")
assert preparar("Este año renové mi cuenta.") == "Este año renové mi cuenta."
assert "No" in preparar("No, quiero cancelar mi cuenta.")

completados.add(3)

Tabla antes | después: 
     id                                                                    text_original                                                                   text_processed     estado           motivo
AUD-001                            Olvidé  la  contraseña  y  necesito  restablecerla.                                     Olvidé la contraseña y necesito restablecerla.  conservar               ok
AUD-010                                       La  sesión  caducó  mientras  trabajaba.                                               La sesión caducó mientras trabajaba.  conservar               ok
AUD-013                                                                                                                                                                   cuarentena    texto ausente
AUD-019                                  El  portal  rechaza  mi  nombre  de  usuario.                                            El portal rechaza mi nombre de usuario.  conservar    

### Tu análisis

La función `preparar()` aplica NFC y colapsa espacios redundantes, pero conserva acentos, ñ, mayúsculas, puntuación y negación. Las pruebas confirman que:
- `"No quiero cancelar mi cuenta."` no cambia. 
- `"sí"` y `"si"` se conservan distintos: el acento distingue significado y no debe eliminarse.
- `"Este año renové mi cuenta."` tampoco cambia: la `ñ` y el acento sobreviven. 
- Los textos ausentes o vacíos van a cuarentena. 
- Los duplicados por clave se marcan con `id_equivalente`. 
En la tabla antes/después solo cambian casos con espacios redundantes o representación descompuesta. El resto se conserva intacto.

**Caso propio:** 
`"No, quiero cancelar mi cuenta."` conserva la coma. Esto muestra que mi regla de colapsar espacios no elimina puntuación: si se eliminara la coma, el sentido cambiaría. La regla respeta la puntuación como señal.

## 4. Interpretar tokenización

Selecciona dos textos del corpus con rasgos distintos (por ejemplo, puntuación y moneda, o un correo y un acento). Usa el tokenizador de español de spaCy visto en clase. Muestra sus tokens y compara al menos una de esas salidas con dividir por espacios.

Explica qué unidades aparecen, una diferencia relevante y por qué importaría en el caso de soporte. Elige ejemplos para analizar, no solo una llamada a la biblioteca. No hay que lematizar, hacer stemming ni entrenar nada.

Antecedentes: Clase 2, demostración del paso 8.

In [14]:
# 1. Tokenizador de español
import spacy
nlp_reglas = spacy.blank("es")
nlp_reglas.add_pipe("sentencizer")

# 2. Elegir dos textos del corpus con rasgos distintos y que cumplan con las especificaciones
texto_1 = corpus.loc[corpus["text_original"].str.contains(r"\$", regex=True, na=False), "text_original"].iloc[0]
texto_2 = corpus.loc[corpus["text_original"].str.contains("@", regex=True, na=False), "text_original"].iloc[0]

print("Texto 1:", texto_1)
print("Texto 2:", texto_2)

# 3. Tokenizar con spaCy
for nombre, texto in [("Texto 1", texto_1), ("Texto 2", texto_2)]:
    doc = nlp_reglas(texto)
    print(f"\n--- {nombre} ---")
    print("Tokens spaCy :", [t.text for t in doc])
    print("Tokens split :", texto.split())
    print("Oraciones    :", [s.text for s in doc.sents])
    
completados.add(4)


Texto 1: Pagué $1,250.50 y la factura muestra $1,520.50. ¿Pueden revisarla?
Texto 2: Envíen la factura a persona@example.invalid; el cargo sigue pendiente.

--- Texto 1 ---
Tokens spaCy : ['Pagué', '$', '1,250.50', 'y', 'la', 'factura', 'muestra', '$', '1,520.50', '.', '¿', 'Pueden', 'revisarla', '?']
Tokens split : ['Pagué', '$1,250.50', 'y', 'la', 'factura', 'muestra', '$1,520.50.', '¿Pueden', 'revisarla?']
Oraciones    : ['Pagué $1,250.50 y la factura muestra $1,520.50. ¿', 'Pueden revisarla?']

--- Texto 2 ---
Tokens spaCy : ['Envíen', 'la', 'factura', 'a', 'persona@example.invalid', ';', 'el', 'cargo', 'sigue', 'pendiente', '.']
Tokens split : ['Envíen', 'la', 'factura', 'a', 'persona@example.invalid;', 'el', 'cargo', 'sigue', 'pendiente.']
Oraciones    : ['Envíen la factura a persona@example.invalid; el cargo sigue pendiente.']


### Tu análisis

Elegí dos textos del corpus con rasgos distintos: 

- El Texto 1 combina moneda, números con formato y signos de apertura (¿). 

Con `spacy.blank("es")` y `sentencizer`, `spaCy` separa la puntuación como tokens propios y mantiene unidos los montos, mientras que `split()` por espacios deja $1,250.50 como un solo token y arrastra la puntuación al final de la palabra

- El Texto 2 contiene un correo electrónico y un punto y coma. 

`spaCy` separa el ';' del correo, pero split() lo deja pegado a este. Esta diferencia puede llegar a invalidar el correo si se uso tal cual lo recibe la máquina. 

En general, para smbis casos estas diferencias importan en soporte porque los montos son señal para pagos y el correo es un dato sensible que conviene aislar. También se reconoce ¿ y ? como unidades separadas, lo que ayuda a segmentar oraciones correctamente.

## 5. Separar y verificar

Separa el corpus conservado en train, validation y test con una semilla declarada y proporción razonable; por ejemplo 70/15/15 u 80/10/10. Separa familias completas, no filas independientes. Explica el propósito de cada partición. Documenta las proporciones reales por filas y por familias y la presencia de categorías; el redondeo puede impedir porcentajes exactos.

Para construir el reparto con operaciones conocidas: prepara una tabla con una fila por familia, ordena sus identificadores, mézclala con sample(frac=1, random_state=tu_semilla), decide los tamaños enteros y asigna a cada grupo una partición. Después incorpora esa asignación a las filas mediante map. Si agrupas por categoría, primero resuelve o aparta familias con etiquetas contradictorias. No se evalúa conocer una API nueva de scikit-learn.

Comprueba: IDs únicos; cada familia en una sola partición; ninguna clave de texto equivalente elegida ni text_processed idéntico cruza particiones; tres particiones no vacías; cobertura de categorías o limitación justificada. Si la preparación crea colisiones nuevas entre familias, revisa la política, agrupa esos casos relacionados o apártalos de forma documentada antes de dividir. No pruebes semillas hasta ocultar un problema.

En una copia, introduce un solapamiento de familia y muestra que tu control lo detecta, sin modificar el corpus de entrega. Explica por qué el mismo caso en train y test produciría una evaluación demasiado optimista. No calcules rendimiento ni uses test para escoger reglas por desempeño.

Antecedentes: Clase 1, sample del paso 2; Clase 2, pasos 10–11. El procedimiento de asignación se proporciona aquí como apoyo; se evalúan la decisión de agrupar y su validación.

In [15]:
# 1. Preparar tabla por familia, incluyendo limpieza
conflictos = corpus[corpus["estado"] == "conservar"].groupby("family")["label"].nunique()
familias_conflictivas = conflictos[conflictos > 1].index.tolist()
print("Familias con etiquetas contradictorias:", familias_conflictivas)

mask_conf = corpus["family"].isin(familias_conflictivas) & (corpus["estado"] == "conservar")
corpus.loc[mask_conf, "estado"] = "cuarentena"
corpus.loc[mask_conf, "motivo"] = "familia con etiquetas contradictorias"

registro = corpus[["id", "estado", "motivo", "id_equivalente"]].rename(columns={"id": "id_original"})
conservados = corpus[corpus["estado"] == "conservar"].copy()

familias = (
    conservados[["family", "label"]]
    .drop_duplicates()
    .sort_values("family")
    .reset_index(drop=True)
)

# 2. Mezclar con semilla declarada
familias_mezcladas = familias.sample(frac=1, random_state=42).reset_index(drop=True)

n = len(familias_mezcladas)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

familias_mezcladas["split"] = (
    ["train"] * n_train +
    ["validation"] * n_val +
    ["test"] * (n - n_train - n_val)
)

# 3. Asignar split a cada fila
mapa_split = dict(zip(familias_mezcladas["family"], familias_mezcladas["split"]))
conservados["split"] = conservados["family"].map(mapa_split)

# 4. Verificar
def validar_split(tabla):
    assert tabla["id"].is_unique, "IDs duplicados"
    assert (tabla.groupby("family")["split"].nunique() == 1).all(), "Familia en varias particiones"
    assert set(tabla["split"]) == {"train", "validation", "test"}, "Faltan particiones"
    fuga = tabla.groupby("text_processed")["split"].nunique()
    assert (fuga == 1).all(), f"Texto equivalente cruzando particiones: {fuga[fuga > 1]}"
    return True

assert validar_split(conservados)

resumen = conservados.groupby("split").agg(
    filas=("id", "size"),
    familias=("family", "nunique"),
)
resumen["prop_filas"] = (resumen["filas"] / resumen["filas"].sum()).round(3)
resumen["prop_familias"] = (resumen["familias"] / resumen["familias"].sum()).round(3)
print(resumen)

print("\nCobertura de label por split:")
cobertura = conservados.groupby(["split", "label"]).size().unstack(fill_value=0)
print(cobertura)
print("\nProporción por split:")
print(cobertura.div(cobertura.sum(axis=1), axis=0).round(3))

# 5. Prueba de fuga controlada (en copia)
bad = conservados.copy()
familia_obj = bad.loc[bad["split"] == "train", "family"].iloc[0]
idx = bad[bad["family"] == familia_obj].index[0]
bad.loc[idx, "split"] = "test"
print("Familia movida:", familia_obj)
try:
    validar_split(bad)
    print("ERROR: no se detectó la fuga")
except AssertionError as e:
    print("Fuga detectada:", e)
assert validar_split(conservados)

completados.add(5)

Familias con etiquetas contradictorias: []
            filas  familias  prop_filas  prop_familias
split                                                 
test           16         6       0.180          0.194
train          62        21       0.697          0.677
validation     11         4       0.124          0.129

Cobertura de label por split:
label       acceso  pagos
split                    
test             8      8
train           32     30
validation       2      9

Proporción por split:
label       acceso  pagos
split                    
test         0.500  0.500
train        0.516  0.484
validation   0.182  0.818
Familia movida: F001
Fuga detectada: Familia en varias particiones


### Tu análisis

Separé el corpus conservado (89 filas, 31 familias) en train/validation/test con semilla 42 y proporción 70/15/15. El reparto se hizo sobre una tabla con una fila por familia, mezclada con `sample(frac=1, random_state=42)`, y se asignó a cada registro con map. Antes de dividir, aparté familias con etiquetas contradictorias (ninguna en este corpus) y verifiqué que text_processed no cruzara particiones.

**Propósito de cada partición:**

- train (62 filas, 21 familias): ajustar el futuro clasificador.
- validation (11 filas, 4 familias): ajustar hiperparámetros y monitorear sin tocar test.
- test (16 filas, 6 familias): evaluación final, una sola vez.

| split | filas | familias | prop_filas | prop_familias |
|---|---|---|---|---|
| train | 62 | 21 | 0.697 | 0.677 |
| validation | 11 | 4 | 0.124 | 0.129 |
| test | 16 | 6 | 0.180 | 0.194 |

El redondeo impide 70/15/15 exacto: con 31 familias, 21/4/6 es lo más cercano.

**Validación:**

`validar_split` confirma: IDs únicos, cada familia en una sola partición, tres particiones no vacías y ninguna clave `text_processed` repetida cruzando particiones. En una copia moví una familia de train a test y el control lo detectó ("Familia en varias particiones"), dejando intacto el corpus de entrega.


## 6. Exportar, reconstruir y explicar

Guarda corpus_preparado.csv con id, text_original, text_processed, label, family y split, en UTF-8. Guarda registro_decisiones.csv para todos los IDs iniciales, con estado y motivo. Los originales nulos pueden exportarse con el marcador `__NULO__`; documenta la misma convención al recargar.

Exporta train.csv, validation.csv y test.csv automáticamente desde la columna split. Son los archivos de partición pedidos por el plan; no los edites por separado. Añade manifiesto.json con versiones de Python, pandas y spaCy, semilla, política y nombre del archivo fuente. El diccionario de nuevas columnas y el reporte de decisiones van dentro del notebook.

Reabre tus CSV y verifica IDs, originales, particiones y conteos. Reinicia el kernel y ejecuta todo sin depender de celdas antiguas ni rutas de tu computadora. Conserva salidas e interpretación y descarga tus archivos si usas Colab.

Responde con evidencia de tu corpus: 1) ¿Por qué conservar o descartar «no» cambia la tarea? 2) ¿Qué se pierde al eliminar acentos y ñ? 3) ¿Por qué conservar el original? 4) ¿Por qué revisar NFC antes de declarar duplicados? 5) ¿Qué riesgo existe si una familia aparece en train y test? 6) ¿Qué limita tu tratamiento de datos sensibles?

Antecedentes: Clase 2, pasos 10–13.

In [16]:
import hashlib

OUT = Path.cwd() / "entrega"
OUT.mkdir(exist_ok=True)

# 1. Sincronizar corpus con registro y asegurar columna split 

if "registro" in dir():
    mapa_estado = dict(zip(registro["id_original"], registro["estado"]))
    mapa_motivo = dict(zip(registro["id_original"], registro["motivo"]))
    corpus["estado"] = corpus["id"].map(mapa_estado).fillna(corpus["estado"])
    corpus["motivo"] = corpus["id"].map(mapa_motivo).fillna(corpus["motivo"])
else:
    registro = corpus[["id", "estado", "motivo", "id_equivalente"]].rename(
        columns={"id": "id_original"}
    )

if "split" not in corpus.columns or corpus["split"].isna().all():
    conservados_final = corpus[corpus["estado"] == "conservar"].copy()
    familias_validas = conservados_final[["family"]].drop_duplicates().sort_values("family")
    familias_mezcladas = familias_validas.sample(frac=1, random_state=42).reset_index(drop=True)
    n = len(familias_mezcladas)
    n_train = int(n * 0.70)
    n_val   = int(n * 0.15)
    familias_mezcladas["split"] = (
        ["train"] * n_train +
        ["validation"] * n_val +
        ["test"] * (n - n_train - n_val)
    )
    mapa_split = dict(zip(familias_mezcladas["family"], familias_mezcladas["split"]))
    corpus["split"] = corpus["family"].map(mapa_split)

corpus.loc[corpus["estado"] != "conservar", "split"] = None

# 2. Guardar corpus preparado 
corpus.to_csv(OUT / "corpus_preparado.csv", index=False, encoding="utf-8")

# 3. Guardar registro de decisiones 
registro.to_csv(OUT / "registro_decisiones.csv", index=False, encoding="utf-8")

# 4. Guardar particiones automáticamente desde la columna split
for nombre in ["train", "validation", "test"]:
    tabla = corpus[corpus["split"] == nombre]
    tabla.to_csv(OUT / f"{nombre}.csv", index=False, encoding="utf-8")

# 5. Manifiesto 
manifiesto = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "spacy": spacy.__version__,
    "semilla": 42,
    "politica": "NFC y espacios; conservar original, negación, acentos y split",
    "fuente": "corpus_auditoria_v2.csv",
    "sha256": hashlib.sha256((OUT / "corpus_preparado.csv").read_bytes()).hexdigest(),
}
(OUT / "manifiesto.json").write_text(
    json.dumps(manifiesto, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(manifiesto)

# 6. Reabrir y verificar
recargado = pd.read_csv(
    OUT / "corpus_preparado.csv",
    encoding="utf-8",
    keep_default_na=False,
    na_values=["__NULO__"],
)

assert set(recargado["id"]) == set(corpus["id"]), "IDs no coinciden"

splits_presentes = set(
    s for s in recargado["split"].unique()
    if isinstance(s, str) and s.strip() != ""
)
assert splits_presentes == {"train", "validation", "test"}, \
    f"Faltan: { {'train','validation','test'} - splits_presentes }"

def normalizar_texto(serie):
    return (
        serie
        .astype("string")
        .replace({"": pd.NA, "__NULO__": pd.NA})
        .fillna("__NULO__")
        .reset_index(drop=True)
    )

orig_corpus    = normalizar_texto(corpus["text_original"])
orig_recargado = normalizar_texto(recargado["text_original"])
assert orig_corpus.equals(orig_recargado), "text_original cambió"

print("Reconstrucción verificada.")
print("\nConteo por split (solo conservados):")
print(recargado[recargado["split"].notna() & (recargado["split"] != "")].groupby("split").size())
print("\nConteo por estado:")
print(recargado["estado"].value_counts())

completados.add(6)

{'python': '3.12.10', 'pandas': '3.0.5', 'spacy': '3.8.16', 'semilla': 42, 'politica': 'NFC y espacios; conservar original, negación, acentos y split', 'fuente': 'corpus_auditoria_v2.csv', 'sha256': 'b76a45bf8b15cbb8d7794e581eb2bb34e89eeca8fcf3d590dde9e08741d2eecc'}
Reconstrucción verificada.

Conteo por split (solo conservados):
split
test          16
train         62
validation    11
dtype: int64

Conteo por estado:
estado
conservar     89
cuarentena     5
duplicado      3
Name: count, dtype: int64


### Tu análisis

**Problema observado:** 

El corpus tiene faltantes, duplicados y familias con etiquetas contradictorias. Además hay representaciones Unicode distintas para el mismo texto.

**Decisión:**
- Aplicar NFC + colapso de espacios. 
- Conservar original. 
- Enmascarar correos/URLs. 
- Poner en cuarentena faltantes, textos insuficientes y familias con etiquetas contradictorias. 
- Separar por familia con semilla 42.

**Justificación y evidencia:**

1. **¿Por qué conservar o descartar "no" cambia la tarea?**
   Conservar "no" mantiene la negación, por ende mantiene la intención del mesanje, lo que es clave para el entendimiento y resolución de problemas. 
   Evidencia: `assert "No" in preparar("No, quiero cancelar mi cuenta.")`.

2. **¿Qué se pierde al eliminar acentos y ñ?**
   Se pierde distinción de significado. 
   Evidencia: `assert preparar("sí") != preparar("si")`.

3. **¿Por qué conservar el original?**
   Permite auditar transformaciones y reproducir el flujo. Evidencia:
   `corpus["text_original"]` intacto tras NFC/espacios.

4. **¿Por qué revisar NFC antes de declarar duplicados?**
   Sin NFC, `informacio\u0301n` y `información` no comparan igual. Visualmente pueden verse similares, pero en cuanto al código que lee la computadora son distintos. 
   Evidencia: AUD-081 cambia con NFC, la clave explícita reveló 8 filas
   duplicadas vs 4 sin NFC.

5. **¿Qué riesgo existe si una familia aparece en train y test?**
   El modelo memorizaría variantes casi idénticas y la evaluación sería
   optimista. Evidencia: prueba de fuga controlada detectada por `validar_split`.

6. **¿Qué limita tu tratamiento de datos sensibles?**
   El regex no cubre todos los formatos ni anonimiza nombres o teléfonos.
   Evidencia: `patron_correo` solo reconoce `usuario@dominio.tld`.

**Limitaciones:** validation puede quedar sin alguna categoría por el tamaño
de las familias. 

## Una sola entrega
Entrega un ZIP con el notebook completado y ejecutado y la carpeta entrega/ que genera tu código. Esa carpeta contiene corpus_preparado.csv, registro_decisiones.csv, train.csv, validation.csv, test.csv y manifiesto.json. No se exige un reporte o README adicional: decisiones, diccionario ampliado, análisis, límites y orden de ejecución quedan en el notebook.

Escribe explicaciones breves, aproximadamente 350–550 palabras en total además de tablas y código; es una orientación, no un límite de calificación. Declara integrantes, aportaciones y ayuda recibida. No entregues capturas como sustituto del código. No incluyas paquetes instalados ni archivos del profesor.

## Criterios de aceptación y checklist
- [x] Auditoría con conteos verificables y casos examinados.
- [x] Problema, decisión, justificación y comprobación conectados.
- [x] Caso NFC del corpus demostrado; acentos y ñ distinguidos de su representación.
- [x] Textos originales e IDs conservados; exclusiones y duplicados trazables.
- [x] Seis comparaciones antes/después y pruebas de español y negación.
- [x] Dos textos tokenizados e interpretación de la comparación.
- [x] Familias y textos equivalentes sin solapamientos; fallo controlado detectado.
- [x] CSV reabiertos y notebook ejecutado desde un kernel nuevo.
- [x] Privacidad, limitaciones, versiones y semilla documentadas.
- [x] ZIP con los archivos indicados y sin información personal real.

Una ejecución de la plantilla vacía solo comprueba preparación y carga; no equivale a resolver la actividad. La celda final avisa si todavía quedan apartados pendientes.

## Rúbrica pública: 100 puntos internos

Se conservan los cinco criterios y pesos del plan de evaluación definitivo. Las nuevas precisiones explicitan la evidencia, sin aumentar el valor de la actividad. Niveles: sobresaliente 100% del criterio; competente 75%; parcial 50%; insuficiente 0%. La contribución a la materia es puntos internos × 0.08.

### Auditoría y procedencia - 20 puntos

- Sobresaliente: Identifica procedencia y columnas; distingue nulo, vacío y espacios; cuenta duplicados indicando el denominador y revisa familias, etiquetas y ejemplos verificables.

- Competente: Cubre estructura, faltantes y duplicados correctamente; omite un detalle secundario de procedencia o del conteo sin cambiar el diagnóstico central.

- Parcial: Aporta conteos reproducibles, pero confunde tipos de ausencia o duplicado, o deja una categoría importante de problemas sin examinar.

- Insuficiente: No presenta auditoría reproducible o sus afirmaciones contradicen los datos sin comprobación.

### Preservación lingüística - 25 puntos

- Sobresaliente: Explicita operaciones aplicadas y evitadas; demuestra NFC con el corpus, conserva original y sentido en español, prueba negación/acentos/ñ y explica dos tokenizaciones.

- Competente: La política y el caso NFC son correctos y conservan señales clave; falta una comparación secundaria o explicar una diferencia de tokenización.

- Parcial: Aplica una preparación parcialmente válida, pero no prueba NFC sobre el corpus o destruye señales de español sin justificar su impacto.

- Insuficiente: No implementa política verificable o elimina sistemáticamente señales esenciales sin evidencia de comprensión.

### Etiquetas y particiones - 25 puntos

- Sobresaliente: Documenta dudas de etiqueta sin adivinar; separa grupos completos con semilla y proporciones razonadas, verifica IDs/textos/familias/cobertura y detecta la fuga introducida.

- Competente: El reparto y las decisiones de etiqueta son válidos, con controles centrales sin fuga; falta un detalle secundario de proporciones o cobertura.

- Parcial: Existe reparto reproducible, pero cruza familias o textos, no detecta la contaminación, o resuelve etiquetas sin evidencia.

- Insuficiente: No entrega particiones comprobables o no existe un mecanismo de separación y verificación.

### Pruebas y reconstrucción - 20 puntos

- Sobresaliente: Reconstruye desde kernel nuevo con funciones; registra cada ID y motivo; exporta y recarga comprobando originales, conteos y particiones; documenta versiones y política.

- Competente: La reconstrucción y los archivos centrales funcionan; falta un dato secundario del manifiesto o una comprobación no esencial.

- Parcial: Solo una parte es reproducible, hay cambios manuales no reconstruidos, se pierde el original o la recarga no comprueba lo declarado.

- Insuficiente: No se puede reconstruir ningún resultado central con los archivos entregados.

### Argumentación y límites - 10 puntos

- Sobresaliente: Conecta problemas, decisiones y pruebas con ejemplos antes/después; justifica privacidad y orden; reconoce alternativas y limitaciones sin atribuir rendimiento no medido.

- Competente: Las decisiones centrales están justificadas y ligadas a evidencia; un límite o alternativa queda poco desarrollado.

- Parcial: Ofrece razones genéricas sin conectar varias decisiones con ejemplos, o confunde reemplazo de un campo con anonimización total.

- Insuficiente: No interpreta las salidas ni ofrece justificaciones verificables.

Se asigna el nivel más alto respaldado por la evidencia del criterio. Un error crítico no implica cero global ni una resta adicional.

**Declaración de Uso de IA**

Herramienta: DeepSeek

Uso realizado: Asistencia en la modificación y adaptación del código proporcionado por mí. Concretamente se le proporcionó pseudocódigo, secciones de código de las clases e instrucciones específicas sobre los cambios requeridos y su propósito, y se utilizó la herramienta para proponer e implementar dichas modificaciones.

Secciones donde se utilizó: Todas las secciones de la actividad en las que se requirieron modificaciones, implementación o ajustes de código.

Validación realizada por el estudiante: Se revisó el código generado o modificado, verificando que correspondiera con las instrucciones y requerimientos de la actividad, así como su integración con el código proporcionado previamente.

Otro (especifique): La IA se utilizó como herramienta de apoyo y no como sustituto de mi trabajo. Las instrucciones, código base y requerimientos de modificación fueron proporcionados por mí.

Confirmación: Confirmo que revisé, verifiqué y asumo la responsabilidad del contenido final entregado.
